In [1]:
import kagglehub
import os
# Download latest version
path = kagglehub.competition_download('word2vec-nlp-tutorial')

print("Path to competition files:", path)
print(os.listdir(path))

Path to competition files: /kaggle/input/competitions/word2vec-nlp-tutorial
['testData.tsv.zip', 'sampleSubmission.csv', 'unlabeledTrainData.tsv.zip', 'labeledTrainData.tsv.zip']


In [2]:
import pandas as pd  
##导入数据
train = pd.read_csv(
    "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip",
    sep="\t",
    compression="zip"
)

test = pd.read_csv(
    "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip",
    sep="\t",
    compression="zip"
)
##观察数据结构

print(train.shape)
print(test.shape)
print(train.head())

(25000, 3)
(25000, 2)
       id  sentiment                                             review
0  5814_8          1  With all this stuff going down at the moment w...
1  2381_9          1  \The Classic War of the Worlds\" by Timothy Hi...
2  7759_3          0  The film starts with a manager (Nicholas Bell)...
3  3630_4          0  It must be assumed that those who praised this...
4  9495_8          1  Superbly trashy and wondrously unpretentious 8...


In [3]:
from bs4 import BeautifulSoup
#导入bs4库，帮助移除html标签，清洗掉html连接，可以得到纯文本数据
raw_review = "This movie was <br /><br /> really good!"#一个评论例子

clean_review = BeautifulSoup(
    raw_review, "html.parser"
).get_text()

print(clean_review)

This movie was  really good!


In [4]:
#现在将句子中标点符号，链接，数字等移除
import re
def review_to_words(review):
    text = BeautifulSoup(
        review, "html.parser"
    ).get_text(" ")

    text = text.lower()
    words = re.findall(r"[a-z]+", text)
    return " ".join(words)
example="this Movie is <br /><br /> really good! 10/10"
key=review_to_words(example)
print(key.split())#分割成单词


['this', 'movie', 'is', 'really', 'good']


In [5]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
stop_words=set(stopwords.words("english"))
print(list(stop_words)[:20])
#引入停用词表，方便去除停用词


['a', 'own', 'wouldn', 'after', 'were', 'mustn', 'with', 'during', 'am', 'for', 'herself', 'while', 'which', "mightn't", 'ours', 'more', 'we', 'in', 'she', "you're"]


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
data="i dont like this movie,no,no,this"
words=review_to_words(data)
def remove_stopwords(words):
    return [w for w in words if w not in stop_words or w in["not","nor","no"]]
keys=words.split()
print(remove_stopwords(keys))
n=len(train["review"])
def clean_text(data):
    clean_data=review_to_words(data)
    clean_data=clean_data.split()
    clean_data=remove_stopwords(clean_data)
    ans=" ".join(clean_data)
    return ans
clean_reviews=[]
# 清理训练集评论
for i in range(n):
    clean_reviews.append(clean_text(train["review"][i]))
print(clean_reviews[0])




['dont', 'like', 'movie', 'no', 'no']
stuff going moment mj started listening music watching odd documentary watched wiz watched moonwalker maybe want get certain insight guy thought really cool eighties maybe make mind whether guilty innocent moonwalker part biography part feature film remember going see cinema originally released subtle messages mj feeling towards press also obvious message drugs bad kay visually impressive course michael jackson unless remotely like mj anyway going hate find boring may call mj egotist consenting making movie mj fans would say made fans true really nice actual feature film bit finally starts minutes excluding smooth criminal sequence joe pesci convincing psychopathic powerful drug lord wants mj dead bad beyond mj overheard plans nah joe pesci character ranted wanted people know supplying drugs etc dunno maybe hates mj music lots cool things like mj turning car robot whole speed demon sequence also director must patience saint came filming kiddy bad s

In [7]:
#建立词袋模型
from sklearn.feature_extraction.text import CountVectorizer
vectorizer=CountVectorizer(
    max_features=5000,
)

x=vectorizer.fit_transform(clean_reviews)
y=train["sentiment"]
print(x.shape)
print(y.shape)


(25000, 5000)
(25000,)


In [8]:
#使用随机森林进行分类
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

xtrain,xval,ytrain,yval=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
model=RandomForestClassifier(n_estimators=100,
                             random_state=42,
                             n_jobs=-1,
                             )
model.fit(xtrain,ytrain)
print(model.score(xval,yval))


0.8476


In [9]:
from sklearn.metrics import roc_auc_score
valid_pred=model.predict_proba(xval)[:,1]
auc=roc_auc_score(yval,valid_pred)
print(auc)


0.92157776


In [10]:
# 对测试集进行处理
n=len(test["review"])
test_reviews=test["review"]
test_data=[]
for i in range(n):
    test_data.append(clean_text(test_reviews[i]))
test_data=vectorizer.transform(test_data)


In [11]:
test_pred=model.predict(test_data)
result=pd.DataFrame({"id":test["id"],"sentiment":test_pred})
result.to_csv("result.csv",index=False)



In [12]:
#word2vec准备
import csv
def review_to_words(review):
    data=BeautifulSoup(review,"html.parser").get_text()
    data=re.sub("[^a-zA-Z]"," ",data)
    data=data.lower()
    data=data.split()
    return data
clean_reviews=[]
train1=pd.read_csv(
    "/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip",
    sep="\t",
    compression="zip",
    engine="python",
    quoting=csv.QUOTE_NONE,
    on_bad_lines="error"
)
for review in train1["review"]:
    clean_reviews.append(review_to_words(review))
print(clean_reviews[0][:10])


['watching', 'time', 'chasers', 'it', 'obvious', 'that', 'it', 'was', 'made', 'by']


In [13]:
!pip install -q gensim

In [14]:
from gensim.models import Word2Vec

In [15]:
#训练模型
model=Word2Vec(
    sentences=clean_reviews,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=5,
    seed=42
)

In [16]:
print(model.wv["good"])#查看good的向量

[-0.1060397   0.10148232  0.12202467  0.1687056  -0.14359756  0.58320415
 -0.06500072  0.65565026 -0.67525995  0.03485929 -0.02888684 -0.20780388
 -0.02868882  0.21508068 -0.03565497  0.00605633  0.25859177 -0.19054848
  0.07785838 -0.17646669 -0.4603522   0.09712595  0.49075362  0.16029087
  0.50689423  0.11320032 -0.38669372 -0.07806626 -0.18791765 -0.24126132
  0.25997534 -0.08378031 -0.5050579  -0.0544657  -0.22121303  0.03554123
 -0.10954153 -0.7145988   0.26170066  0.4127126   0.29889867 -0.07341801
  0.29577893 -0.33706895 -0.43433705  0.23202796  0.16104068 -0.16058594
  0.18616259 -0.10769626 -0.19119944 -0.5609305  -0.41022867  0.04519201
  0.28250962  0.13553946 -0.16354744 -0.04040413  0.0035931   0.49316496
 -0.13596764  0.06013725 -0.29547808  0.17711927  0.19671279 -0.18071248
  0.31390312 -0.09621368 -0.21891744  0.2644962  -0.02789757 -0.33538827
 -0.0831347  -0.0490499  -0.00780458  0.14428847 -0.02851924 -0.3793842
 -0.2030867   0.07350847 -0.0364071  -0.22398624 -0.

In [17]:
#查找语言相近的词
print(model.wv.most_similar("good", topn=10))

[('decent', 0.8226720690727234), ('great', 0.8133962750434875), ('bad', 0.8028015494346619), ('fine', 0.7754056453704834), ('nice', 0.7614834308624268), ('terrific', 0.7211951613426208), ('solid', 0.7103505730628967), ('believeable', 0.7101083993911743), ('serviceable', 0.7018550038337708), ('farfetched', 0.700031578540802)]


In [18]:
import numpy as np
def review_to_vector(review):
    vectors=[model.wv[word] for word in review if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    mean_vector=np.mean(vectors,axis=0)
    return mean_vector
x=np.stack(
    [review_to_vector(review) for review in clean_reviews]
    
)
print(x.shape)



(50000, 100)


In [19]:
#清洗文本，拆分句子
import nltk
import re
from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize



In [20]:
nltk.download("punkt")
nltk.download("punkt_tab")
def review_to_sentences(reviews):
    review_text=BeautifulSoup(reviews,"html.parser").get_text()
    raw_sentences=sent_tokenize(review_text.strip())
    sencences=[]
    for sentence in raw_sentences:
        letters=re.sub(r"[^a-zA-Z]", " ", sentence)
        letters=letters.lower()
        words=letters.split()
        if words:
            sencences.append(words)
    return sencences


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [21]:
sentences=[]
review_all=train["review"].tolist()+train1["review"].tolist()
for review in review_all:
    sentences.extend(review_to_sentences(review))

In [22]:
print("句子数量：",len(sentences))
print(sentences[0])

句子数量： 792953
['with', 'all', 'this', 'stuff', 'going', 'down', 'at', 'the', 'moment', 'with', 'mj', 'i', 've', 'started', 'listening', 'to', 'his', 'music', 'watching', 'the', 'odd', 'documentary', 'here', 'and', 'there', 'watched', 'the', 'wiz', 'and', 'watched', 'moonwalker', 'again']


In [23]:
#使用label和unlabeled数据训练word2vec模型
from gensim.models import Word2Vec

num_features = 300
min_word_count = 40
num_workers = 4
context = 10
downsampling = 1e-3

model = Word2Vec(
    sentences=sentences,
    vector_size=num_features,
    window=context,
    min_count=min_word_count,
    workers=num_workers,
    sample=downsampling,
    sg=1,
    seed=42,
    epochs=5
)

print("词表大小：", len(model.wv))
print("good 的向量维度：", model.wv["good"].shape)
print(model.wv.most_similar("good", topn=10))

词表大小： 16490
good 的向量维度： (300,)
[('great', 0.626554012298584), ('decent', 0.617763340473175), ('nice', 0.5523861646652222), ('bad', 0.5497524738311768), ('solid', 0.5176432132720947), ('fine', 0.5052576065063477), ('terrific', 0.5017476677894592), ('cool', 0.4735037088394165), ('excellent', 0.47302812337875366), ('serviceable', 0.45873069763183594)]


In [24]:
#评论向量研究
#先导入停用词
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
stop_words=set(stopwords.words("english"))


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [25]:
#评论清洗
import re
from bs4 import BeautifulSoup
def review_to_wordlist(review, remove_stopwords=False):
    review=BeautifulSoup(review,"html.parser").get_text()
    review=re.sub("[^a-zA-Z]"," ",review)
    review=review.lower().split()
    if remove_stopwords:
        review=[w for w in review if w not in stop_words]
    return review


In [26]:
import numpy as np
num_features=model.vector_size
def makeFeatureVec(review,model,num_features):
    featureVec=np.zeros(num_features,dtype=np.float32)
    nwords=0
    for w in review:
        if w in model.wv:
            nwords+=1
            featureVec+=model.wv[w]
    if nwords>0:
        featureVec/=nwords
    return featureVec

In [27]:
#获得评论向量矩阵
def getAvgFeatureVecs(reviews, model, num_features):
    result=np.zeros((len(reviews),num_features),dtype=np.float32)
    for i,review in enumerate(reviews):
        Vec=makeFeatureVec(review,model,num_features)
        result[i]=Vec
    return result

In [28]:
#清洗训练集数据
clean_train_reviews = []

for review in train["review"]:
    clean_train_reviews.append(
        review_to_wordlist(
            review,
            remove_stopwords=True
        )
    )

#获得训练集评论向量矩阵
trainDataVecs = getAvgFeatureVecs(
    clean_train_reviews,
    model,
    num_features
)
#处理测试集
clean_test_reviews = []

for review in test["review"]:
    clean_test_reviews.append(
        review_to_wordlist(
            review,
            remove_stopwords=True
        )
    )

testDataVecs = getAvgFeatureVecs(
    clean_test_reviews,
    model,
    num_features
)


In [29]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

forest.fit(
    trainDataVecs,
    train["sentiment"]
)


RandomForestClassifier(n_jobs=-1, random_state=42)

In [30]:
result = forest.predict(testDataVecs)

output = pd.DataFrame({
    "id": test["id"].values,
    "sentiment": result
})

output.to_csv(
    "/kaggle/working/Word2Vec_AverageVectors.csv",
    index=False
)

print(output.head())
print("提交文件形状：", output.shape)
print("类别分布：")
print(output["sentiment"].value_counts())

assert len(output) == len(test)
assert output["id"].tolist() == test["id"].tolist()
assert output["id"].is_unique

         id  sentiment
0  12311_10          1
1    8348_2          0
2    5828_4          1
3    7186_2          0
4   12128_7          1
提交文件形状： (25000, 2)
类别分布：
sentiment
1    12538
0    12462
Name: count, dtype: int64


In [33]:
#使用kmeans进行分类
import numpy as np

word_vectors = model.wv.vectors.astype("float32")
vocab_words = model.wv.index_to_key

print("词向量矩阵形状：", word_vectors.shape)
print("词表大小：", len(vocab_words))
print("词向量维度：", model.vector_size)

词向量矩阵形状： (16490, 300)
词表大小： 16490
词向量维度： 300


In [34]:
from sklearn.cluster import MiniBatchKMeans
num_clusters=min(500,max(2,len(vocab_words)//2))
kmeans_cluster=MiniBatchKMeans(
    n_clusters=num_clusters,
    batch_size=4096,
    n_init=10,
    max_iter=100,
    random_state=42,
    verbose=10
)
idx=kmeans_cluster.fit_predict(word_vectors)
print(idx.shape)
print(num_clusters)


Init 1/10 with method k-means++
Inertia for init 1/10: 110777.796875
Init 2/10 with method k-means++
Inertia for init 2/10: 111000.625
Init 3/10 with method k-means++
Inertia for init 3/10: 110455.625
Init 4/10 with method k-means++
Inertia for init 4/10: 103924.4609375
Init 5/10 with method k-means++
Inertia for init 5/10: 103909.796875
Init 6/10 with method k-means++
Inertia for init 6/10: 110144.5
Init 7/10 with method k-means++
Inertia for init 7/10: 111111.1875
Init 8/10 with method k-means++
Inertia for init 8/10: 104208.8203125
Init 9/10 with method k-means++
Inertia for init 9/10: 109695.3203125
Init 10/10 with method k-means++
Inertia for init 10/10: 109968.703125
[MiniBatchKMeans] Reassigning 494 cluster centers.
Minibatch step 1/402: mean batch inertia: 8.441832542419434
Minibatch step 2/402: mean batch inertia: 7.960913181304932, ewa inertia: 7.960913181304932
[MiniBatchKMeans] Reassigning 493 cluster centers.
Minibatch step 3/402: mean batch inertia: 7.824951171875, ewa in

In [35]:
#建立词与簇编号的字典
word_centroid_map = dict(zip(vocab_words, idx))

In [36]:
from collections import defaultdict

cluster_words = defaultdict(list)

for word, cluster_id in word_centroid_map.items():
    if len(cluster_words[cluster_id]) < 15:
        cluster_words[cluster_id].append(word)

for cluster_id in range(10):
    print(f"\n簇 {cluster_id}:")
    print(cluster_words[cluster_id])


簇 0:
['tiresome']

簇 1:
['studded']

簇 2:
['lights']

簇 3:
['sparse']

簇 4:
['tomei', 'marisa']

簇 5:
['gotta']

簇 6:
['intervention']

簇 7:
['madison']

簇 8:
['longer']

簇 9:
['flopped']


In [37]:
def create_bag_of_centroids(wordlist, word_centroid_map, num_clusters):
    bag_of_centroids=np.zeros(num_clusters,dtype="float32")
    for word in wordlist:
        if word in word_centroid_map:
            cluster_id=word_centroid_map[word]
            bag_of_centroids[cluster_id]+=1
    return bag_of_centroids

In [38]:
#将训练集和测试集转换为聚类计数矩阵
train_centroids=np.zeros((len(clean_train_reviews),num_clusters),dtype="float32")
for i in range(len(clean_train_reviews)):
    train_centroids[i]=create_bag_of_centroids(clean_train_reviews[i],word_centroid_map,num_clusters)

test_centroids = np.zeros((len(clean_test_reviews), num_clusters),dtype="float32")

for i, review in enumerate(clean_test_reviews):
    test_centroids[i] = create_bag_of_centroids(
        review,
        word_centroid_map,
        num_clusters
    )
print("运行结束")


运行结束


In [39]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

forest_centroids = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

forest_centroids.fit(
    train_centroids,
    train["sentiment"]
)

result_centroids = forest_centroids.predict(test_centroids)

output_centroids = pd.DataFrame({
    "id": test["id"].values,
    "sentiment": result_centroids
})

output_centroids.to_csv(
    "/kaggle/working/BagOfCentroids.csv",
    index=False
)